# UT2 · 2 — Pandas en profundidad

Segundo cuaderno de la Unidad 2. Aquí nos centramos en **Pandas**: crear y explorar un DataFrame, seleccionar y filtrar datos, agruparlos y combinarlos, y limpiarlos antes de usarlos.

Es autocontenido: genera su propio dataset de ejemplo (estadísticas de jugadores de un videojuego online, con nulos y duplicados incluidos a propósito) para poder practicar sin depender de ningún archivo externo.

In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)
print("Pandas cargado correctamente. Versión:", pd.__version__)

Pandas cargado correctamente. Versión: 3.0.2


## 1 · Crear un DataFrame

Lo más habitual en clase es `pd.read_csv("archivo.csv")`, pero aquí generamos los datos en memoria a partir de un diccionario — la misma estructura que tendría cualquier CSV una vez cargado.

In [2]:
plataformas = ["PC", "Consola"]
n = 40

df = pd.DataFrame({
    "jugador_id": range(1, n + 1),
    "plataforma": np.random.choice(plataformas, size=n),
    "horas_jugadas": np.round(np.random.normal(6, 2, size=n), 1),
    "puntuacion": np.round(np.random.normal(6.2, 1.8, size=n), 1),
})

# Ensuciamos el dataset a propósito, como pasa con datos reales
df.loc[[3, 15, 27], "puntuacion"] = np.nan          # algunos nulos
df.loc[[5, 22], "horas_jugadas"] = np.nan
df = pd.concat([df, df.iloc[[0, 1]]], ignore_index=True)  # dos filas duplicadas
df["puntuacion"] = df["puntuacion"].clip(0, 10)

df.head()

,jugador_id,plataforma,horas_jugadas,puntuacion
0,1,PC,4.0,4.7
1,2,Consola,6.6,5.6
2,3,PC,4.2,6.8
3,4,PC,3.2,NaN
4,5,PC,8.9,5.3


### Limpiar los nombres Y los valores de texto: espacios de más

Un CSV real casi nunca viene con nombres de columna ni valores de texto perfectos — a veces traen espacios de más al principio o al final (`"  Ana  "`, `"Nombre "`), que a simple vista no se notan pero rompen comparaciones y agrupaciones. Antes de trabajar con un dataset nuevo, es buena costumbre limpiarlo a lo primero.

In [3]:
df_sucio_ejemplo = pd.DataFrame({
    "  Nombre ": ["  Ana  ", "Luis", " Marta "],
    "Edad": [23, 31, 29],
})
print("Columnas antes: ", list(df_sucio_ejemplo.columns))
print("Primer nombre, con comillas para ver los espacios:", repr(df_sucio_ejemplo["  Nombre "].iloc[0]))

df_sucio_ejemplo.columns = df_sucio_ejemplo.columns.str.strip()          # limpia los NOMBRES de columna
df_sucio_ejemplo["Nombre"] = df_sucio_ejemplo["Nombre"].str.strip()      # limpia los VALORES de esa columna

print("\nColumnas después:", list(df_sucio_ejemplo.columns))
print("Primer nombre limpio, con comillas:", repr(df_sucio_ejemplo["Nombre"].iloc[0]))

Columnas antes:  ['  Nombre ', 'Edad']
Primer nombre, con comillas para ver los espacios: '  Ana  '

Columnas después: ['Nombre', 'Edad']
Primer nombre limpio, con comillas: 'Ana'


En nuestro `df` de jugadores los nombres ya vienen limpios, pero no lo asumas nunca de entrada con un CSV que no has creado tú — esta comprobación (`df.columns`, y un vistazo con `repr()` a algún valor de texto) debería ser uno de tus primeros pasos siempre.

## 2 · Tipos de dato y conversión con astype

In [4]:
print("Tipos de cada columna:")
print(df.dtypes)

df["plataforma"] = df["plataforma"].astype("category")   # menos memoria con pocos valores distintos
print("\nTras convertir plataforma a category:", df["plataforma"].dtype)

Tipos de cada columna:
jugador_id         int64
plataforma           str
horas_jugadas    float64
puntuacion       float64
dtype: object

Tras convertir plataforma a category: category


## 3 · Renombrar columnas y el índice

In [5]:
df_renombrado = df.rename(columns={"horas_jugadas": "horas_semana"})
print(df_renombrado.columns.tolist())

df_indexado = df.set_index("jugador_id")   # jugador_id pasa a ser el índice
print(df_indexado.head(2))

df_indexado.reset_index().head(2)          # deshace el cambio: jugador_id vuelve a ser columna

['jugador_id', 'plataforma', 'horas_semana', 'puntuacion']
           plataforma  horas_jugadas  puntuacion
jugador_id                                      
1                  PC            4.0         4.7
2             Consola            6.6         5.6


,jugador_id,plataforma,horas_jugadas,puntuacion
0,1,PC,4.0,4.7
1,2,Consola,6.6,5.6


## 4 · Exploración inicial

Antes de tocar nada, siempre lo mismo: cuántas filas y columnas hay, de qué tipo es cada una, y si hay huecos.

In [6]:
print("Forma del dataset (filas, columnas):", df.shape)
df.info()

Forma del dataset (filas, columnas): (42, 4)
<class 'pandas.DataFrame'>
RangeIndex: 42 entries, 0 to 41
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   jugador_id     42 non-null     int64   
 1   plataforma     42 non-null     category
 2   horas_jugadas  40 non-null     float64 
 3   puntuacion     39 non-null     float64 
dtypes: category(1), float64(2), int64(1)
memory usage: 1.2 KB


In [7]:
df.describe()

,jugador_id,horas_jugadas,puntuacion
count,42.000000,40.000000,39.000000
mean,19.595238,5.592500,5.992308
std,12.115420,1.796491,1.627298
min,1.000000,2.100000,1.500000
25%,9.250000,4.150000,5.100000
50%,19.500000,5.600000,5.900000
75%,29.750000,6.625000,6.900000
max,40.000000,9.700000,9.000000


## 5 · Cuatro formas de seleccionar datos, y filtrado booleano

`df["col"]` para una columna, `.loc` por etiqueta, `.iloc` por posición y `.query()` para condiciones más legibles.

In [8]:
print(df["puntuacion"].head(3))                                        # una columna
print(df.loc[0:2, ["plataforma", "puntuacion"]])                       # .loc: por etiqueta (filas 0 a 2 incluida)
print(df.iloc[0:3, 0:2])                                               # .iloc: por posición
print(df.query("puntuacion >= 8 and plataforma == \'PC\'").head())      # .query(): condición como texto

0    4.7
1    5.6
2    6.8
Name: puntuacion, dtype: float64
  plataforma  puntuacion
0         PC         4.7
1    Consola         5.6
2         PC         6.8
   jugador_id plataforma
0           1         PC
1           2    Consola
2           3         PC
    jugador_id plataforma  horas_jugadas  puntuacion
11          12         PC            6.8         8.0


In [9]:
# Filtrado con máscaras booleanas
alta_habilidad = df[df["puntuacion"] >= 7]
pc_alta_habilidad = df[(df["plataforma"] == "PC") & (df["puntuacion"] >= 7)]

print("Total filas:        ", len(df))
print("Alta habilidad:     ", len(alta_habilidad))
print("PC y alta habilidad:", len(pc_alta_habilidad))

Total filas:         42
Alta habilidad:      9
PC y alta habilidad: 5


### Dos atajos que verás constantemente en código ajeno: isin() y assign()

In [10]:
# isin(): comprobar pertenencia a una lista de valores, en vez de encadenar muchos "or"
solo_estas_plataformas = df[df["plataforma"].isin(["PC", "Consola"])]
print("Filas con plataforma en la lista:", len(solo_estas_plataformas))

# assign(): crea una o varias columnas nuevas y devuelve una COPIA, sin tocar el DataFrame original
df_extra = df.assign(
    horas_por_dia=df["horas_jugadas"] / 7,
    es_top=df["puntuacion"] >= 8,
)
df_extra[["jugador_id", "horas_jugadas", "horas_por_dia", "puntuacion", "es_top"]].head()

Filas con plataforma en la lista: 42


,jugador_id,horas_jugadas,horas_por_dia,puntuacion,es_top
0,1,4.0,0.571429,4.7,False
1,2,6.6,0.942857,5.6,False
2,3,4.2,0.600000,6.8,False
3,4,3.2,0.457143,NaN,False
4,5,8.9,1.271429,5.3,False


**Ejercicio:** sin mirar la solución, escribe el código para quedarte solo con los jugadores que tengan `puntuacion` por encima de 8 **o** más de 9 horas jugadas, usando una máscara booleana con `|` (el "or" de Pandas).

In [11]:
# Solución del ejercicio
destacados = df[(df["puntuacion"] > 8) | (df["horas_jugadas"] > 9)]
print("Jugadores destacados por nota o por horas:", len(destacados))
destacados[["jugador_id", "puntuacion", "horas_jugadas"]].head()

Jugadores destacados por nota o por horas: 4


,jugador_id,puntuacion,horas_jugadas
9,10,8.6,6.2
15,16,NaN,9.7
17,18,9.0,3.9
26,27,8.9,5.8


## 6 · apply(): tu propia función, celda a celda o fila a fila

Cuando ninguna función incluida en Pandas hace exactamente lo que necesitas, `apply()` te permite aplicar cualquier función (o `lambda`) columna a columna, o fila a fila con `axis=1`.

In [12]:
def clasificar(puntuacion):
    if pd.isna(puntuacion):
        return "sin dato"
    elif puntuacion >= 7:
        return "alta"
    elif puntuacion >= 4:
        return "media"
    else:
        return "baja"

df["categoria"] = df["puntuacion"].apply(clasificar)
df[["jugador_id", "puntuacion", "categoria"]].head()

,jugador_id,puntuacion,categoria
0,1,4.7,media
1,2,5.6,media
2,3,6.8,media
3,4,NaN,sin dato
4,5,5.3,media


In [13]:
# axis=1 aplica la función fila a fila, útil cuando el resultado depende de varias columnas a la vez
df["horas_y_nota"] = df.apply(
    lambda fila: f"{fila['horas_jugadas']}h / nota {fila['puntuacion']}",
    axis=1,
)
df[["jugador_id", "horas_y_nota"]].head(3)

,jugador_id,horas_y_nota
0,1,4.0h / nota 4.7
1,2,6.6h / nota 5.6
2,3,4.2h / nota 6.8


### Reemplazar y mapear valores

`.replace()` cambia valores concretos sin tocar el resto; `.map()` aplica una correspondencia completa (como un diccionario) a toda una columna — muy habitual para convertir códigos en etiquetas más legibles para un informe.

In [14]:
# map(): sustituye CADA valor según el diccionario (si falta una clave, el resultado es NaN)
df["plataforma_legible"] = df["plataforma"].map({"PC": "Ordenador", "Consola": "Videoconsola"})
print(df[["plataforma", "plataforma_legible"]].head())

# replace(): cambia solo los valores puntuales que le indiques, deja el resto tal cual
df["categoria_es"] = df["categoria"].replace({"sin dato": "desconocido"})
df["categoria_es"].value_counts()

  plataforma plataforma_legible
0         PC          Ordenador
1    Consola       Videoconsola
2         PC          Ordenador
3         PC          Ordenador
4         PC          Ordenador


categoria_es
media          27
alta            9
desconocido     3
baja            3
Name: count, dtype: int64

## 7 · Ordenar filas

In [15]:
print(df.sort_values("puntuacion", ascending=False).head(3))   # los 3 primeros, de mayor a menor

print("\nTop 5 por nlargest (más directo que ordenar toda la tabla):")
print(df.nlargest(5, "puntuacion")[["jugador_id", "plataforma", "puntuacion"]])

    jugador_id plataforma  horas_jugadas  puntuacion categoria  \
17          18    Consola            3.9         9.0      alta   
26          27    Consola            5.8         8.9      alta   
9           10    Consola            6.2         8.6      alta   

       horas_y_nota plataforma_legible categoria_es  
17  3.9h / nota 9.0       Videoconsola         alta  
26  5.8h / nota 8.9       Videoconsola         alta  
9   6.2h / nota 8.6       Videoconsola         alta  

Top 5 por nlargest (más directo que ordenar toda la tabla):
    jugador_id plataforma  puntuacion
17          18    Consola         9.0
26          27    Consola         8.9
9           10    Consola         8.6
11          12         PC         8.0
35          36         PC         7.9


## 8 · value_counts(): contar categorías de un vistazo

In [16]:
print("Jugadores por plataforma:")
print(df["plataforma"].value_counts())

print("\nLo mismo en proporciones (normalize=True):")
print(df["plataforma"].value_counts(normalize=True).round(2))

print("\nLo mismo por categoría de puntuación:")
print(df["categoria"].value_counts())

Jugadores por plataforma:
plataforma
Consola    21
PC         21
Name: count, dtype: int64

Lo mismo en proporciones (normalize=True):
plataforma
Consola    0.5
PC         0.5
Name: proportion, dtype: float64

Lo mismo por categoría de puntuación:
categoria
media       27
alta         9
sin dato     3
baja         3
Name: count, dtype: int64


## 9 · Agrupar y combinar datasets

`groupby` resume el dataset por categorías, y `merge` lo combina con otra tabla relacionada.

In [17]:
resumen_plataforma = df.groupby("plataforma").agg(
    puntuacion_media=("puntuacion", "mean"),
    total_jugadores=("jugador_id", "count"),
)
resumen_plataforma

,puntuacion_media,total_jugadores
plataforma,,
Consola,5.985,21
PC,6.000,21


### Agrupar por varias claves, pivot_table y transform

In [18]:
df["nivel"] = pd.cut(df["puntuacion"], bins=[0, 4, 7, 10], labels=["bajo", "medio", "alto"])

print("Media por plataforma y nivel:")
print(df.groupby(["plataforma", "nivel"])["puntuacion"].mean())

print("\nMisma idea en formato tabla cruzada (pivot_table):")
print(df.pivot_table(values="puntuacion", index="nivel", columns="plataforma", aggfunc="mean"))

# transform: pega el resultado del grupo de vuelta en cada fila, sin colapsar el DataFrame
df["media_de_su_plataforma"] = df.groupby("plataforma")["puntuacion"].transform("mean")
df[["jugador_id", "plataforma", "puntuacion", "media_de_su_plataforma"]].head()

Media por plataforma y nivel:
plataforma  nivel
Consola     bajo     2.050000
            medio    5.857143
            alto     8.400000
PC          bajo     3.800000
            medio    5.608333
            alto     7.820000
Name: puntuacion, dtype: float64

Misma idea en formato tabla cruzada (pivot_table):
plataforma   Consola        PC
nivel                         
bajo        2.050000  3.800000
medio       5.857143  5.608333
alto        8.400000  7.820000


,jugador_id,plataforma,puntuacion,media_de_su_plataforma
0,1,PC,4.7,6.000
1,2,Consola,5.6,5.985
2,3,PC,6.8,6.000
3,4,PC,NaN,6.000
4,5,PC,5.3,6.000


In [19]:
# Una segunda tabla relacionada, para practicar merge
analistas = pd.DataFrame({
    "plataforma": ["PC", "Consola"],
    "analista": ["Marina", "Javier"],
})

df_con_analista = df.merge(analistas, on="plataforma", how="left")
df_con_analista[["jugador_id", "plataforma", "analista"]].head()

,jugador_id,plataforma,analista
0,1,PC,Marina
1,2,Consola,Javier
2,3,PC,Marina
3,4,PC,Marina
4,5,PC,Marina


### Merge con indicator, y cuidado con las claves duplicadas

In [20]:
# indicator=True añade una columna "_merge" que dice de dónde viene cada fila: útil para depurar un merge
comprobacion = df.merge(analistas, on="plataforma", how="outer", indicator=True)
print(comprobacion["_merge"].value_counts())

# validate="one_to_one" hace que Pandas lance un error si la clave no es única en alguna tabla,
# en vez de multiplicar filas en silencio si hay valores repetidos
df.merge(analistas, on="plataforma", how="left", validate="many_to_one").shape

_merge
both          42
left_only      0
right_only     0
Name: count, dtype: int64


(42, 11)

### concat: apilar dos DataFrames verticalmente

Muy habitual cuando los datos llegan repartidos en varios archivos (por ejemplo, un CSV por semana) y hay que juntarlos en uno solo.

In [21]:
jugadores_semana_2 = pd.DataFrame({
    "jugador_id": range(41, 46),
    "plataforma": np.random.choice(plataformas, size=5),
    "horas_jugadas": np.round(np.random.normal(6, 2, size=5), 1),
    "puntuacion": np.round(np.random.normal(6.2, 1.8, size=5), 1),
})

df_todo = pd.concat([df[["jugador_id", "plataforma", "horas_jugadas", "puntuacion"]], jugadores_semana_2], ignore_index=True)
print("Filas antes:", len(df), " · filas después de concat:", len(df_todo))

Filas antes: 42  · filas después de concat: 47


## 10 · Limpieza y preprocesamiento

Detectamos y resolvemos, uno a uno, los problemas típicos: columnas redundantes, nulos, duplicados y una columna categórica sin codificar.

### Detectar columnas redundantes o muy correlacionadas

Antes de limpiar nulos, vale la pena preguntarse si sobra alguna columna — por ejemplo, dos columnas casi idénticas (typo habitual al fusionar fuentes de datos) no aportan información nueva.

In [22]:
# .corr() calcula la correlación entre TODAS las columnas numéricas a la vez
correlaciones = df[["horas_jugadas", "puntuacion", "media_de_su_plataforma"]].corr()
correlaciones.round(2)
# una correlación cercana a 1 (fuera de la diagonal) es señal de columnas redundantes: podría bastar con una de las dos

,horas_jugadas,puntuacion,media_de_su_plataforma
horas_jugadas,1.00,-0.18,-0.04
puntuacion,-0.18,1.00,0.00
media_de_su_plataforma,-0.04,0.00,1.00


In [23]:
print("Nulos por columna antes de limpiar:")
print(df.isna().sum())

Nulos por columna antes de limpiar:
jugador_id                0
plataforma                0
horas_jugadas             2
puntuacion                3
categoria                 0
horas_y_nota              0
plataforma_legible        0
categoria_es              0
nivel                     3
media_de_su_plataforma    0
dtype: int64


In [24]:
# Duplicados
print("Filas duplicadas:", df.duplicated().sum())

df_limpio = df.drop_duplicates().copy()
print("Filas tras eliminar duplicados:", len(df_limpio))

Filas duplicadas: 2
Filas tras eliminar duplicados: 40


In [25]:
# Nulos: rellenamos con la media de cada columna en vez de eliminar filas
# (con solo 40 jugadores, perder filas nos deja con muy pocos datos)
df_limpio["puntuacion"] = df_limpio["puntuacion"].fillna(df_limpio["puntuacion"].mean())
df_limpio["horas_jugadas"] = df_limpio["horas_jugadas"].fillna(df_limpio["horas_jugadas"].mean())

print("Nulos por columna después de limpiar:")
print(df_limpio.isna().sum())

Nulos por columna después de limpiar:
jugador_id                0
plataforma                0
horas_jugadas             0
puntuacion                0
categoria                 0
horas_y_nota              0
plataforma_legible        0
categoria_es              0
nivel                     3
media_de_su_plataforma    0
dtype: int64


In [26]:
# Codificación de la variable categórica "plataforma"
df_codificado = pd.get_dummies(df_limpio, columns=["plataforma"])
df_codificado.head()

,jugador_id,horas_jugadas,puntuacion,categoria,horas_y_nota,plataforma_legible,categoria_es,nivel,media_de_su_plataforma,plataforma_Consola,plataforma_PC
0,1,4.0,4.700000,media,4.0h / nota 4.7,Ordenador,media,medio,6.000,False,True
1,2,6.6,5.600000,media,6.6h / nota 5.6,Videoconsola,media,medio,5.985,True,False
2,3,4.2,6.800000,media,4.2h / nota 6.8,Ordenador,media,medio,6.000,False,True
3,4,3.2,6.037838,sin dato,3.2h / nota nan,Ordenador,desconocido,NaN,6.000,False,True
4,5,8.9,5.300000,media,8.9h / nota 5.3,Ordenador,media,medio,6.000,False,True


**Reto rápido:** ¿por qué hemos rellenado los nulos con la media en vez de eliminar esas filas con `dropna()`? Coméntalo en clase antes de seguir — no hay una única respuesta correcta, depende de cuántos datos tengas y de qué tan importantes sean esas filas.

## 11 · Guardar el resultado

Una vez limpio, lo normal es guardar el dataset para no repetir el proceso cada vez.

In [27]:
df_limpio.to_csv("jugadores_limpio.csv", index=False)
print("Guardado. Primeras filas del archivo:")
pd.read_csv("jugadores_limpio.csv").head(3)

Guardado. Primeras filas del archivo:


,jugador_id,plataforma,horas_jugadas,puntuacion,categoria,horas_y_nota,plataforma_legible,categoria_es,nivel,media_de_su_plataforma
0,1,PC,4.0,4.7,media,4.0h / nota 4.7,Ordenador,media,medio,6.000
1,2,Consola,6.6,5.6,media,6.6h / nota 5.6,Videoconsola,media,medio,5.985
2,3,PC,4.2,6.8,media,4.2h / nota 6.8,Ordenador,media,medio,6.000


## 12 · Practica por tu cuenta: publicaciones de redes sociales

Mismo tipo de herramientas, tema distinto — para que veas que Pandas no es solo para videojuegos. Aquí tienes un dataset (inventado, y sucio a propósito) de publicaciones sobre series de TV en redes sociales.

In [28]:
datos_redes = {
    "usuario": ["  martasocial", "cinefilo_99", "   ", "juan.tv", "ana_streaming  ", "pop_corner"],
    "serie": ["Stranger Things", "La Casa de Papel", "Wednesday", "The Bear", "Stranger Things", "Wednesday"],
    "red": ["Instagram", "TikTok", "Instagram", "Instagram", "TikTok", "Instagram"],
    "seguidores": [15400, np.nan, 8200, 23000, np.nan, 5100],
    "valoracion": ["Buena", "  Excelente", "mala", "", "Buena  ", "Excelente"],
}

df_redes = pd.DataFrame(datos_redes)
df_redes

,usuario,serie,red,seguidores,valoracion
0,martasocial,Stranger Things,Instagram,15400.0,Buena
1,cinefilo_99,La Casa de Papel,TikTok,NaN,Excelente
2,,Wednesday,Instagram,8200.0,mala
3,juan.tv,The Bear,Instagram,23000.0,
4,ana_streaming,Stranger Things,TikTok,NaN,Buena
5,pop_corner,Wednesday,Instagram,5100.0,Excelente


**Ejercicio:** sin mirar la solución, escribe el código para dejar `df_redes` limpio y responder a estas preguntas:

1. Quita los espacios de más en `usuario` y en `valoracion`, y pon `valoracion` en minúsculas.
2. Rellena los `seguidores` que faltan con la mediana de la columna.
3. Elimina la fila donde `usuario` queda vacío tras la limpieza.
4. ¿Cuántos seguidores de media tiene cada serie (`groupby`)?
5. Usando `.map()`, crea una columna `valoracion_num` que convierta "excelente" → 3, "buena" → 2, "mala" → 1.

In [29]:
# Solución del ejercicio
df_redes["usuario"] = df_redes["usuario"].str.strip()
df_redes["valoracion"] = df_redes["valoracion"].str.strip().str.lower()

df_redes["seguidores"] = df_redes["seguidores"].fillna(df_redes["seguidores"].median())

df_redes = df_redes[df_redes["usuario"] != ""]

media_seguidores_por_serie = df_redes.groupby("serie")["seguidores"].mean()

df_redes["valoracion_num"] = df_redes["valoracion"].map({"excelente": 3, "buena": 2, "mala": 1})

print("Seguidores medios por serie:")
print(media_seguidores_por_serie)
print()
df_redes

Seguidores medios por serie:
serie
La Casa de Papel    11800.0
Stranger Things     13600.0
The Bear            23000.0
Wednesday            5100.0
Name: seguidores, dtype: float64



,usuario,serie,red,seguidores,valoracion,valoracion_num
0,martasocial,Stranger Things,Instagram,15400.0,buena,2.0
1,cinefilo_99,La Casa de Papel,TikTok,11800.0,excelente,3.0
3,juan.tv,The Bear,Instagram,23000.0,,NaN
4,ana_streaming,Stranger Things,TikTok,11800.0,buena,2.0
5,pop_corner,Wednesday,Instagram,5100.0,excelente,3.0


---
**Resumen de lo practicado:** crear un DataFrame · limpiar nombres de columna y valores de texto con `.str.strip()` · tipos de dato con `astype` · exploración (`head`, `info`, `describe`) · selección con `.loc`/`.iloc`/`.query`/`isin`/`assign` · `apply`, `.map()` y `.replace()` · ordenar y `value_counts` · `groupby`, `pivot_table`, `transform` · `merge` y `concat` · limpieza (columnas redundantes, nulos, duplicados, `get_dummies`) · exportar a CSV · un caso práctico completo con datos de redes sociales.

**Siguiente cuaderno:** `UT2_03_matplotlib.ipynb`, para visualizar este mismo dataset.